# NYC Urban Infrastructure & Service Analytics
## Power BI Preparation

This notebook prepares validated analytical datasets and KPI tables for Power BI dashboard development.

### Objectives
- Prepare dashboard-ready datasets
- Create KPI summary tables
- Prepare geographic analysis tables
- Prepare agency and complaint performance tables
- Prepare socioeconomic analysis tables
- Export clean Power BI source files

### Analysis Period
January 1, 2025 – December 31, 2025

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import pandas as pd
import os

BASE_PATH = "/content/drive/MyDrive/NYC_311_2025_BACKUP"

final_parquet = os.path.join(BASE_PATH, "NYC_311_2025_FINAL.parquet")

df = pd.read_parquet(final_parquet)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


Dataset loaded successfully.
Shape: (3655040, 19)

Columns:
['service_request_id', 'created_at', 'closed_at', 'resolution_hours', 'resolution_days', 'resolution_quality', 'agency', 'complaint_type', 'descriptor', 'status', 'resolution_description', 'incident_zip', 'borough', 'community_board', 'community_board_number', 'community_board_borough', 'latitude', 'longitude', 'geo_quality_final']


In [4]:
kpi_summary = pd.DataFrame({
    "Metric": [
        "Total Service Requests",
        "Unique Service Requests",
        "Agencies",
        "Complaint Types",
        "Community Boards",
        "Boroughs",
        "Valid Resolution Records",
        "Average Resolution Hours",
        "Resolved Within 24 Hours (%)"
    ],
    "Value": [
        len(df),
        df["service_request_id"].nunique(),
        df["agency"].nunique(),
        df["complaint_type"].nunique(),
        df.loc[
            df["community_board"].notna() &
            (df["community_board"].astype(str).str.strip() != ""),
            "community_board"
        ].nunique(),
        df.loc[
            df["borough"].notna() &
            (df["borough"].astype(str).str.strip() != ""),
            "borough"
        ].nunique(),
        df.loc[
            df["resolution_hours"].notna() &
            (df["resolution_hours"] >= 0),
            "resolution_hours"
        ].count(),
        round(
            df.loc[
                df["resolution_hours"].notna() &
                (df["resolution_hours"] >= 0),
                "resolution_hours"
            ].mean(),
            2
        ),
        round(
            (
                (
                    df["resolution_hours"].notna() &
                    (df["resolution_hours"] >= 0) &
                    (df["resolution_hours"] <= 24)
                ).sum()
                /
                (
                    df["resolution_hours"].notna() &
                    (df["resolution_hours"] >= 0)
                ).sum()
                * 100
            ),
            2
        )
    ]
})

kpi_summary


,Metric,Value
0,Total Service Requests,3655040.00
1,Unique Service Requests,3655040.00
2,Agencies,15.00
3,Complaint Types,874.00
4,Community Boards,59.00
5,Boroughs,5.00
6,Valid Resolution Records,3525111.00
7,Average Resolution Hours,234.61
8,Resolved Within 24 Hours (%),62.18


In [5]:
monthly_demand = (
    df.assign(month=df["created_at"].dt.to_period("M").astype(str))
      .groupby("month")
      .agg(
          service_requests=("service_request_id", "count")
      )
      .reset_index()
)

monthly_demand


,month,service_requests
0,2025-01,348180
1,2025-02,255364
2,2025-03,281221
3,2025-04,272549
4,2025-05,295058
5,2025-06,306442
6,2025-07,315883
7,2025-08,304039
8,2025-09,302684
9,2025-10,336612


In [6]:
agency_performance = (
    df.groupby("agency")
      .agg(
          service_requests=("service_request_id", "count"),
          avg_resolution_hours=(
              "resolution_hours",
              lambda x: x[x >= 0].mean()
          ),
          resolved_within_24h_pct=(
              "resolution_hours",
              lambda x: (x[x >= 0].le(24).sum() / x[x >= 0].notna().sum() * 100)
              if x[x >= 0].notna().sum() > 0 else 0
          )
      )
      .reset_index()
)

agency_performance["avg_resolution_hours"] = (
    agency_performance["avg_resolution_hours"].round(2)
)

agency_performance["resolved_within_24h_pct"] = (
    agency_performance["resolved_within_24h_pct"].round(2)
)

agency_performance = agency_performance.sort_values(
    "service_requests",
    ascending=False
)

agency_performance


,agency,service_requests,avg_resolution_hours,resolved_within_24h_pct
11,New York City Police Department,1717594,2.98,98.94
6,Department of Housing Preservation and Develop...,774535,367.99,13.56
8,Department of Sanitation,326780,89.75,38.67
9,Department of Transportation,212103,242.69,39.46
3,Department of Environmental Protection,197143,91.13,53.63
7,Department of Parks and Recreation,110855,1241.87,22.78
0,Department of Buildings,103066,1315.21,25.40
4,Department of Health and Mental Hygiene,83396,500.25,48.73
5,Department of Homeless Services,51449,113.28,72.32
14,Taxi and Limousine Commission,36225,2320.26,8.67


In [7]:
complaint_performance = (
    df.groupby("complaint_type")
      .agg(
          service_requests=("service_request_id", "count"),
          avg_resolution_hours=(
              "resolution_hours",
              lambda x: x[x >= 0].mean()
          ),
          resolved_within_24h_pct=(
              "resolution_hours",
              lambda x: (
                  (x[x >= 0].le(24).sum() / x[x >= 0].notna().sum() * 100)
                  if x[x >= 0].notna().sum() > 0 else 0
              )
          )
      )
      .reset_index()
)

complaint_performance["avg_resolution_hours"] = (
    complaint_performance["avg_resolution_hours"].round(2)
)

complaint_performance["resolved_within_24h_pct"] = (
    complaint_performance["resolved_within_24h_pct"].round(2)
)

complaint_performance = complaint_performance.sort_values(
    "service_requests",
    ascending=False
)

print("Complaint types:", len(complaint_performance))

complaint_performance.head(15)


Complaint types: 874


,complaint_type,service_requests,avg_resolution_hours,resolved_within_24h_pct
373,Illegal Parking,564577,3.12,99.05
493,Noise - Residential,456599,3.31,98.50
336,HEAT/HOT WATER,315265,42.29,27.94
65,Blocked Driveway,169126,3.54,98.67
494,Noise - Street/Sidewalk,168021,1.81,99.77
786,UNSANITARY CONDITION,115996,593.30,2.01
857,Water System,75994,94.64,71.58
548,PLUMBING,68741,550.25,6.47
716,Street Condition,68208,68.81,46.49
8,Abandoned Vehicle,66013,4.98,97.24


In [8]:
borough_performance = (
    df.assign(
        borough_clean=df["borough"].replace("", pd.NA).fillna("UNKNOWN")
    )
    .groupby("borough_clean")
    .agg(
        service_requests=("service_request_id", "count"),
        avg_resolution_hours=(
            "resolution_hours",
            lambda x: x[x >= 0].mean()
        ),
        resolved_within_24h_pct=(
            "resolution_hours",
            lambda x: (
                (x[x >= 0].le(24).sum() / x[x >= 0].notna().sum() * 100)
                if x[x >= 0].notna().sum() > 0 else 0
            )
        )
    )
    .reset_index()
)

borough_performance["avg_resolution_hours"] = (
    borough_performance["avg_resolution_hours"].round(2)
)

borough_performance["resolved_within_24h_pct"] = (
    borough_performance["resolved_within_24h_pct"].round(2)
)

borough_performance = borough_performance.sort_values(
    "service_requests",
    ascending=False
)

borough_performance


,borough_clean,service_requests,avg_resolution_hours,resolved_within_24h_pct
1,BROOKLYN,1003412,201.90,61.36
3,QUEENS,813632,260.89,68.28
0,BRONX,781563,168.56,60.84
2,MANHATTAN,673197,309.63,57.64
5,UNKNOWN,258820,279.39,65.28
4,STATEN ISLAND,124416,260.94,56.23


In [9]:
community_board_performance = (
    df.assign(
        community_board_clean=(
            df["community_board"]
            .replace("", pd.NA)
            .fillna("UNKNOWN")
        )
    )
    .groupby("community_board_clean")
    .agg(
        service_requests=("service_request_id", "count"),
        avg_resolution_hours=(
            "resolution_hours",
            lambda x: x[x >= 0].mean()
        ),
        resolved_within_24h_pct=(
            "resolution_hours",
            lambda x: (
                (x[x >= 0].le(24).sum() / x[x >= 0].notna().sum() * 100)
                if x[x >= 0].notna().sum() > 0 else 0
            )
        )
    )
    .reset_index()
)

community_board_performance["avg_resolution_hours"] = (
    community_board_performance["avg_resolution_hours"].round(2)
)

community_board_performance["resolved_within_24h_pct"] = (
    community_board_performance["resolved_within_24h_pct"].round(2)
)

community_board_performance = community_board_performance.sort_values(
    "service_requests",
    ascending=False
)

print("Community board values:", len(community_board_performance))

community_board_performance.head(20)


Community board values: 60


,community_board_clean,service_requests,avg_resolution_hours,resolved_within_24h_pct
59,UNKNOWN,408445,288.51,63.15
47,12 BRONX,182831,76.68,79.86
49,12 MANHATTAN,101122,217.52,55.72
15,04 BRONX,80797,197.47,49.64
1,01 BROOKLYN,80680,215.35,65.00
20,05 BROOKLYN,77251,171.55,61.23
3,01 QUEENS,76125,200.75,68.26
41,10 MANHATTAN,75107,274.57,49.49
50,12 QUEENS,74873,244.42,62.28
27,07 BRONX,72578,201.59,52.77


In [10]:
socioeconomic_file = os.path.join(
    BASE_PATH,
    "NYC_311_2025_TRACT_SOCIOECONOMIC_ANALYSIS.csv"
)

socioeconomic = pd.read_csv(socioeconomic_file)

print("Socioeconomic table loaded.")
print("Shape:", socioeconomic.shape)
print("\nColumns:")
print(socioeconomic.columns.tolist())


Socioeconomic table loaded.
Shape: (2334, 18)

Columns:
['GEOID', 'total_requests', 'valid_resolution_records', 'resolution_coverage_pct', 'median_resolution_hours', 'p75_resolution_hours', 'p90_resolution_hours', 'median_resolution_days', 'p75_resolution_days', 'p90_resolution_days', 'population', 'requests_per_1000_residents', 'median_household_income', 'poverty_rate_pct', 'unemployment_rate_pct', 'housing_units', 'population_threshold_1000', 'socioeconomic_analysis_eligible']


In [11]:
powerbi_socioeconomic = socioeconomic[
    [
        "GEOID",
        "total_requests",
        "valid_resolution_records",
        "resolution_coverage_pct",
        "median_resolution_hours",
        "p75_resolution_hours",
        "p90_resolution_hours",
        "median_resolution_days",
        "requests_per_1000_residents",
        "population",
        "median_household_income",
        "poverty_rate_pct",
        "unemployment_rate_pct",
        "housing_units",
        "population_threshold_1000",
        "socioeconomic_analysis_eligible"
    ]
].copy()

print("Power BI socioeconomic table created.")
print("Shape:", powerbi_socioeconomic.shape)

powerbi_socioeconomic.head()


Power BI socioeconomic table created.
Shape: (2334, 16)


,GEOID,total_requests,valid_resolution_records,resolution_coverage_pct,median_resolution_hours,p75_resolution_hours,p90_resolution_hours,median_resolution_days,requests_per_1000_residents,population,median_household_income,poverty_rate_pct,unemployment_rate_pct,housing_units,population_threshold_1000,socioeconomic_analysis_eligible
0,36005000100,35,34,97.142857,19.950417,197.075069,1440.828778,0.831267,11.864407,2950,NaN,NaN,NaN,0,True,False
1,36005000200,1572,1538,97.837150,6.499861,28.702917,183.867222,0.270828,310.610551,5061,123729.0,12.882829,6.098032,1484,True,True
2,36005000400,1510,1478,97.880795,7.516667,42.725000,257.291500,0.313194,227.101820,6649,105924.0,4.466837,8.423586,2255,True,True
3,36005001600,1490,1459,97.919463,21.059722,119.256944,608.291167,0.877488,265.549813,5611,52147.0,15.162319,8.852105,2142,True,True
4,36005001901,1860,1831,98.440860,23.338333,140.746667,836.855000,0.972431,823.008850,2260,58083.0,22.453499,10.585745,1160,True,True


In [12]:
POWERBI_PATH = os.path.join(BASE_PATH, "POWERBI_DATA")

os.makedirs(POWERBI_PATH, exist_ok=True)

print("Power BI export folder ready:")
print(POWERBI_PATH)


Power BI export folder ready:
/content/drive/MyDrive/NYC_311_2025_BACKUP/POWERBI_DATA


In [13]:
kpi_file = os.path.join(
    POWERBI_PATH,
    "PowerBI_KPI_Summary.csv"
)

kpi_summary.to_csv(kpi_file, index=False)

print("KPI summary exported successfully.")
print("File:", kpi_file)
print("Rows:", len(kpi_summary))


KPI summary exported successfully.
File: /content/drive/MyDrive/NYC_311_2025_BACKUP/POWERBI_DATA/PowerBI_KPI_Summary.csv
Rows: 9


In [14]:
monthly_file = os.path.join(
    POWERBI_PATH,
    "PowerBI_Monthly_Demand.csv"
)

monthly_demand.to_csv(monthly_file, index=False)

print("Monthly demand exported successfully.")
print("File:", monthly_file)
print("Rows:", len(monthly_demand))
print("Columns:", monthly_demand.columns.tolist())


Monthly demand exported successfully.
File: /content/drive/MyDrive/NYC_311_2025_BACKUP/POWERBI_DATA/PowerBI_Monthly_Demand.csv
Rows: 12
Columns: ['month', 'service_requests']


In [15]:
agency_file = os.path.join(
    POWERBI_PATH,
    "PowerBI_Agency_Performance.csv"
)

agency_performance.to_csv(agency_file, index=False)

print("Agency performance exported successfully.")
print("File:", agency_file)
print("Rows:", len(agency_performance))
print("Columns:", agency_performance.columns.tolist())


Agency performance exported successfully.
File: /content/drive/MyDrive/NYC_311_2025_BACKUP/POWERBI_DATA/PowerBI_Agency_Performance.csv
Rows: 15
Columns: ['agency', 'service_requests', 'avg_resolution_hours', 'resolved_within_24h_pct']


In [16]:
complaint_file = os.path.join(
    POWERBI_PATH,
    "PowerBI_Complaint_Performance.csv"
)

complaint_performance.to_csv(complaint_file, index=False)

print("Complaint performance exported successfully.")
print("File:", complaint_file)
print("Rows:", len(complaint_performance))
print("Columns:", complaint_performance.columns.tolist())


Complaint performance exported successfully.
File: /content/drive/MyDrive/NYC_311_2025_BACKUP/POWERBI_DATA/PowerBI_Complaint_Performance.csv
Rows: 874
Columns: ['complaint_type', 'service_requests', 'avg_resolution_hours', 'resolved_within_24h_pct']


In [17]:
borough_file = os.path.join(
    POWERBI_PATH,
    "PowerBI_Borough_Performance.csv"
)

borough_performance.to_csv(borough_file, index=False)

print("Borough performance exported successfully.")
print("File:", borough_file)
print("Rows:", len(borough_performance))
print("Columns:", borough_performance.columns.tolist())


Borough performance exported successfully.
File: /content/drive/MyDrive/NYC_311_2025_BACKUP/POWERBI_DATA/PowerBI_Borough_Performance.csv
Rows: 6
Columns: ['borough_clean', 'service_requests', 'avg_resolution_hours', 'resolved_within_24h_pct']


In [18]:
community_board_file = os.path.join(
    POWERBI_PATH,
    "PowerBI_Community_Board_Performance.csv"
)

community_board_performance.to_csv(
    community_board_file,
    index=False
)

print("Community board performance exported successfully.")
print("File:", community_board_file)
print("Rows:", len(community_board_performance))
print("Columns:", community_board_performance.columns.tolist())


Community board performance exported successfully.
File: /content/drive/MyDrive/NYC_311_2025_BACKUP/POWERBI_DATA/PowerBI_Community_Board_Performance.csv
Rows: 60
Columns: ['community_board_clean', 'service_requests', 'avg_resolution_hours', 'resolved_within_24h_pct']


In [19]:
socioeconomic_file = os.path.join(
    POWERBI_PATH,
    "PowerBI_Socioeconomic_Analysis.csv"
)

powerbi_socioeconomic.to_csv(
    socioeconomic_file,
    index=False
)

print("Socioeconomic analysis exported successfully.")
print("File:", socioeconomic_file)
print("Rows:", len(powerbi_socioeconomic))
print("Columns:", powerbi_socioeconomic.columns.tolist())


Socioeconomic analysis exported successfully.
File: /content/drive/MyDrive/NYC_311_2025_BACKUP/POWERBI_DATA/PowerBI_Socioeconomic_Analysis.csv
Rows: 2334
Columns: ['GEOID', 'total_requests', 'valid_resolution_records', 'resolution_coverage_pct', 'median_resolution_hours', 'p75_resolution_hours', 'p90_resolution_hours', 'median_resolution_days', 'requests_per_1000_residents', 'population', 'median_household_income', 'poverty_rate_pct', 'unemployment_rate_pct', 'housing_units', 'population_threshold_1000', 'socioeconomic_analysis_eligible']


In [20]:
powerbi_files = [
    "PowerBI_KPI_Summary.csv",
    "PowerBI_Monthly_Demand.csv",
    "PowerBI_Agency_Performance.csv",
    "PowerBI_Complaint_Performance.csv",
    "PowerBI_Borough_Performance.csv",
    "PowerBI_Community_Board_Performance.csv",
    "PowerBI_Socioeconomic_Analysis.csv"
]

print("POWER BI EXPORT VALIDATION")
print("=" * 50)

for filename in powerbi_files:
    filepath = os.path.join(POWERBI_PATH, filename)

    if os.path.exists(filepath):
        check = pd.read_csv(filepath)
        size_mb = os.path.getsize(filepath) / (1024 ** 2)

        print(f"✅ {filename}")
        print(f"   Rows: {len(check):,}")
        print(f"   Columns: {len(check.columns)}")
        print(f"   Size: {size_mb:.2f} MB")
    else:
        print(f"❌ MISSING: {filename}")

print("=" * 50)
print("Power BI export validation complete.")


POWER BI EXPORT VALIDATION
✅ PowerBI_KPI_Summary.csv
   Rows: 9
   Columns: 2
   Size: 0.00 MB
✅ PowerBI_Monthly_Demand.csv
   Rows: 12
   Columns: 2
   Size: 0.00 MB
✅ PowerBI_Agency_Performance.csv
   Rows: 15
   Columns: 4
   Size: 0.00 MB
✅ PowerBI_Complaint_Performance.csv
   Rows: 874
   Columns: 4
   Size: 0.03 MB
✅ PowerBI_Borough_Performance.csv
   Rows: 6
   Columns: 4
   Size: 0.00 MB
✅ PowerBI_Community_Board_Performance.csv
   Rows: 60
   Columns: 4
   Size: 0.00 MB
✅ PowerBI_Socioeconomic_Analysis.csv
   Rows: 2,334
   Columns: 16
   Size: 0.42 MB
Power BI export validation complete.
